# Parametric Map Generation

Generates voxel-wise CEUS parametric maps (PE, AUC, TP, T0) for a single patient scan. A CEUS `.nii.gz` cine and its corresponding VOI segmentation are loaded, then a sliding-window TIC is computed across the VOI using a vectorized lognormal fit. The fitted parameters are saved as `.npy` arrays in a specified output folder for downstream visualization and comparison.

In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print(Path().cwd())
os.chdir(Path(os.getcwd()).parent)
print(Path().cwd())

/Users/samantha/QuantUS-Plugins-CEUS/China_Data
/Users/samantha/QuantUS-Plugins-CEUS


## Setup

Reload modules and set working directory to the repo root.

## Select Contrast-Enhanced Ultrasound (CEUS) Cine and Parser

In [2]:
from src.image_loading.options import get_scan_loaders

print("Available scan loaders:", list(get_scan_loaders().keys()))

Available scan loaders: ['avi', 'nifti', 'custom_dicom', 'mp4']


In [123]:
scan_type = 'nifti'

scan_path = '/Users/samantha/Desktop/tul/china data/p9/new_v1/CEUS-24626 S8 3D-1.nii.gz'
scan_loader_kwargs = {
    'transpose': False,
}

In [124]:
from src.entrypoints import scan_loading_step

image_data = scan_loading_step(scan_type, scan_path, **scan_loader_kwargs)

## Load Segmentation

Assumes same segmentation for each frame

In [125]:
from src.seg_loading.options import get_seg_loaders

print("Available segmentation loaders:", list(get_seg_loaders().keys()))

Available segmentation loaders: ['nifti', 'load_bolus_mask']


In [126]:
seg_type = 'nifti'

seg_path = '/Users/samantha/Desktop/tul/china data/p9/new_v1/v1_new_voi.nii.gz'
seg_loader_kwargs = {}

In [127]:
from src.entrypoints import seg_loading_step

seg_data = seg_loading_step(seg_type, image_data, seg_path, scan_path, **seg_loader_kwargs)

## CEUS Quantitative Temporal Curve Analysis (Parametric Map Mode)

In [128]:
from src.time_series_analysis.options import get_analysis_types, get_required_kwargs

all_analysis_types, all_analysis_funcs = get_analysis_types()
print("Available analysis types:", list(all_analysis_types.keys()))

Available analysis types: ['curves_paramap', 'curves']


In [129]:
# IMPORTANT: Use curves_paramap for parametric map generation
analysis_type = 'curves_paramap'

print("Available analysis functions:", list(all_analysis_funcs.keys()))

Available analysis functions: ['pyradiomics', 'tic']


In [130]:
analysis_funcs = ['tic']

required_kwargs = get_required_kwargs(analysis_type, analysis_funcs)
print("Required kwargs for current analysis:", required_kwargs)

Required kwargs for current analysis: ['ax_vox_len', 'cor_vox_ovrlp', 'sag_vox_ovrlp', 'ax_vox_ovrlp', 'sag_vox_len', 'cor_vox_len']


In [131]:
# Set frame rate
image_data.frame_rate = 1

# Required kwargs for parametric map analysis
analysis_kwargs = {
    'ax_vox_ovrlp': 50.0,
    'sag_vox_ovrlp': 50.0,
    'cor_vox_ovrlp': 50.0,
    'ax_vox_len': 5.0,
    'sag_vox_len': 5.0,
    'cor_vox_len': 5.0,
}

In [132]:
import copy
import numpy as np
from tqdm import tqdm
from src.time_series_analysis.curves_paramap.framework import CurvesParamapAnalysis


class VectorizedCurvesParamapAnalysis(CurvesParamapAnalysis):

    def compute_curves(self):
        data = self.image_data.intensities_for_analysis
        is_3d = data.ndim == 4
        if not is_3d and data.ndim != 3:
            raise ValueError('Image data must be either 2D+time or 3D+time.')

        n_frames = data.shape[3] if is_3d else data.shape[0]

        voxel_vol = float(np.prod(self.image_data.pixdim))

        self.curves = []
        for ix, window in tqdm(enumerate(self.windows), desc='Computing curves', total=len(self.windows)):
            entry = {}
            if is_3d:
                ax_start, sag_start, cor_start, ax_end, sag_end, cor_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Coronal Start Pix'] = cor_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                entry['Window-Coronal End Pix'] = cor_end
                window_data = data[sag_start:sag_end+1, cor_start:cor_end+1, ax_start:ax_end+1, :]
                num_voxels = window_data.shape[0] * window_data.shape[1] * window_data.shape[2]
                means = np.exp(window_data / 24.09).reshape(-1, n_frames).mean(axis=0)
            else:
                ax_start, sag_start, ax_end, sag_end = window
                entry['Window-Axial Start Pix'] = ax_start
                entry['Window-Sagittal Start Pix'] = sag_start
                entry['Window-Axial End Pix'] = ax_end
                entry['Window-Sagittal End Pix'] = sag_end
                window_data = data[:, ax_start:ax_end+1, sag_start:sag_end+1]
                num_voxels = window_data.shape[1] * window_data.shape[2]
                means = np.exp(window_data / 24.09).reshape(n_frames, -1).mean(axis=1)

            entry['TIC'] = means.tolist()
            entry['TIC_vol'] = (means * voxel_vol * num_voxels).tolist()
            self.curves.append(entry)

        if self.curves_output_path:
            self.save_curves()


print('VectorizedCurvesParamapAnalysis defined')

VectorizedCurvesParamapAnalysis defined


### Vectorized Curve Computation

Subclass of `CurvesParamapAnalysis` that computes mean TIC per sliding window using vectorized NumPy operations for speed. Applies the log-linear transform (`exp(pixel / 24.09)`) to convert stored pixel values to linear intensity before averaging.

In [133]:
analyzed_image_data = copy.deepcopy(image_data)

analysis_obj = VectorizedCurvesParamapAnalysis(analyzed_image_data, seg_data, analysis_funcs, **analysis_kwargs)
analysis_obj.compute_curves()

print("Analysis object type:", type(analysis_obj))

Computing curves: 100%|██████████| 1779/1779 [00:01<00:00, 1387.02it/s]

Analysis object type: <class '__main__.VectorizedCurvesParamapAnalysis'>


## Curve Quantification

In [134]:
from src.curve_quantification.options import get_quantification_funcs

quantification_funcs = get_quantification_funcs()
print("Available quantification functions:", quantification_funcs.keys())

Available quantification functions: dict_keys(['auc_no_fit', 'cmus_firstorder', 'dte', 'first_order_full', 'first_order_select', 'lognormal_fit_full', 'lognormal_fit_select', 'wash_rates'])


In [135]:
function_names = ['lognormal_fit_full']  # or [] for all functions
output_path = '/Users/samantha/Desktop/tul/china data/p9/new_v1/new_paramap/whole_output.csv'
curve_quantifications_kwargs = {
    'curves_to_fit': ['TIC', 'TIC_vol'],
    'tic_name': 'TIC'
}

In [136]:
from src.entrypoints import curve_quantification_step

curve_quant = curve_quantification_step(analysis_obj, function_names, output_path, **curve_quantifications_kwargs)

# Verify analysis type
print("curve_quant.analysis_objs type:", type(curve_quant.analysis_objs))

curve_quant.analysis_objs type: <class '__main__.VectorizedCurvesParamapAnalysis'>


## Parametric Map Saving

In [137]:
from src.entrypoints import visualization_step

vis_type = 'paramap'
params = []
vis_funcs = []
vis_kwargs = {
    'paramap_folder_path': '/Users/samantha/Desktop/tul/china data/p9/new_v1/new_paramap',
    'hide_all_visualizations': False,
}

vis_obj = visualization_step(curve_quant, vis_type, params, vis_funcs, **vis_kwargs)